In [10]:

import numpy as np
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from core.Log import *
from core.plots import *
from core.modelUtils import *
from core.CNNmodel import *
from core.CardiacCTdataset import *
from core.preprocessing import *
from core.globals import *
from core.globals import PARAM_GRID
import logging
from core.CVsplits import *
import ast
from tqdm import tqdm
logger = logging.getLogger('root')
fold_log = logging.getLogger('folds')
root_logger()
folds_logger()

CNTRL_dicom_root = "../Takotsubo-Syndrome/data/Inputs/normal_cases/"
TTS_dicom_root = "../Takotsubo-Syndrome/data/Inputs/takotsubo_cases/"
root_dir = "data/cases/"


pools = ["holdout", "main"]

'''
only hypers i care about:
{'paramID': 1,
'learning_rate': 0.0001,
'weight_decay': 1e-05},
'''


main_dataset=load_dataset(pool="main")
all_folds_data=get_fold_stats()
DL = DataLoaderFactory(main_dataset, all_folds_data)
OUTER_FOLDS = 4; INNER_FOLDS = 3

hp_search_results = load_hp_search_results()
completed_tasks = {(res['outer_fold_id'], res['paramID']) for res in hp_search_results}
#if len(hp_search_results) != 0: remaining = [task for task in get_all_tasks() if task not in {(res['outer_fold_id'], res['paramID']) for res in hp_search_results}]
#else: remaining = get_all_tasks()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
No hyperparameter search done.


In [ ]:
OUTER_FOLDS = 4; INNER_FOLDS = 3
main_dataset=load_dataset(pool="main")
all_folds_data=get_fold_stats()
all_hp_search_results = load_hp_search_results()
completed_tasks = {(res['outer_fold_id'], res['paramID']) for res in all_hp_search_results}
DL = DataLoaderFactory(main_dataset, all_folds_data)
completed_tasks = run_OUTER_hp_search(all_hp_search_results, completed_tasks, DL)


No hyperparameter search done.
4-fold Nested CV Hyperparameter Search


Fold 0 | Hyperparameter Search...: 100%|██████████| 9/9 [00:00<00:00, 8994.22it/s]
Fold 1 | Hyperparameter Search...: 100%|██████████| 9/9 [00:00<?, ?it/s]
Fold 2 | Hyperparameter Search...: 100%|██████████| 9/9 [00:00<?, ?it/s]
Fold 3 | Hyperparameter Search...: 100%|██████████| 9/9 [00:00<?, ?it/s]


In [ ]:


		# Average the validation performance for this hyperparameter set across all inner folds
		hyperparameter_performance[str(hypers)] = avg_loss
		logger.info(f"Hyperparams: {hypers} -> Avg Inner Val Loss: {avg_loss:.4f}")

	# 3. SELECT the best hyperparameters for this outer fold
	best_hyperparams_str = min(hyperparameter_performance, key=hyperparameter_performance.get)
	best_hyperparams = ast.literal_eval(best_hyperparams_str) # Safely convert string back to dict
	OUTER_FOLD_BEST_HYPERPARAMS.append(best_hyperparams)

	logger.info(f"Best hyperparameters for Outer Fold {OUT_K + 1}: {best_hyperparams}")
	logger.info("Training final model for this outer fold on all of its training data...")

	# use the last inner-fold's validation set again
	final_train_indices, final_val_indices = list(INNER_cv.split(OUTER_train, OUTER_train_lbls))[-1]
	final_train_data = [OUTER_train[i] for i in final_train_indices]
	final_val_data = [OUTER_train[i] for i in final_val_indices]

	train_dataset = CardiacCTDataset(final_train_data, train_transforms, OUT_K_stats)
	val_dataset   = CardiacCTDataset(final_val_data, val_test_transforms, OUT_K_stats)
	test_dataset   = CardiacCTDataset(OUTER_test, val_test_transforms, OUT_K_stats)

	num_workers = 4
	trn_loader  = DataLoader(train_dataset, batch_size=best_hyperparams["batch_size"], shuffle=True, num_workers=num_workers)
	val_loader  = DataLoader(val_dataset, batch_size=best_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)
	final_test_loader  = DataLoader(test_dataset, batch_size=best_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)

	final_model = MultiViewCNN(dropout_rate=best_hyperparams["DR"])
	trained_model, epoch_history, best_val_loss = outer_train_model(final_model, trn_loader, val_loader, best_hyperparams)

	logger.info("Evaluating final model on the outer test set...")
	final_scores = evaluate_model(trained_model, final_test_loader, best_hyperparams)
	final_scores['outer_fold'] = OUT_K + 1
	final_scores['model'] = 'MultiViewCNN'
	logger.info(f"End of Outer Fold {OUT_K + 1} | Test results ---> {final_scores}")

	OUTER_FOLD_SCORES.append(final_scores)
	OUTER_FOLD_EPOCH_HISTORY.append(epoch_history)

logger.info("\n--- Nested Cross-Validation Complete ---")
logger.info(f"Final scores from all {OUTER_K} folds: {OUTER_FOLD_SCORES}")
# Calculate and log the average performance across all outer folds
avg_accuracy = np.mean([s['accuracy'] for s in OUTER_FOLD_SCORES])
std_accuracy = np.std([s['accuracy'] for s in OUTER_FOLD_SCORES])
avg_auc = np.mean([s['auc'] for s in OUTER_FOLD_SCORES])
std_auc = np.std([s['auc'] for s in OUTER_FOLD_SCORES])

logger.info(f"Average Model Accuracy: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
logger.info(f"Average Model AUC: {avg_auc:.4f} ± {std_auc:.4f}")



In [ ]:

best_fold_index = np.argmax([s['auc'] for s in OUTER_FOLD_SCORES])
best_overall_hyperparams = OUTER_FOLD_BEST_HYPERPARAMS[best_fold_index]
logger.info(f"\n--- Hyperparameter search complete ---")
logger.info(f"Best performing hyperparameters identified: {best_overall_hyperparams}")
final_train_data, final_val_data = train_test_split(
	training_dl,
	test_size=0.1, # Use 10% of the main set for validation
	random_state=42,
	stratify=training_lbls
)
FINAL_stats = get_dataset_stats(final_train_data)
final_test_dl = [d for d in full_dl if d['pool']== "holdout"]
val_test_transforms, train_transforms = get_transforms(FINAL_stats)


train_dataset = CardiacCTDataset(final_train_data, train_transforms, FINAL_stats)
val_dataset   = CardiacCTDataset(final_val_data, val_test_transforms, FINAL_stats)
test_dataset   = CardiacCTDataset(final_test_dl, val_test_transforms, FINAL_stats)

num_workers = 4
trn_loader  = DataLoader(train_dataset, batch_size=best_overall_hyperparams["batch_size"], shuffle=True, num_workers=num_workers)
val_loader  = DataLoader(val_dataset, batch_size=best_overall_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)
final_test_loader  = DataLoader(test_dataset, batch_size=best_overall_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)
